### RC4 Algorithm

In [2]:
def RC4_Init(key):
    # RC4 Initialization - Key Scheduling Algorithm
    S = [0] * 256
    K = [0] * 256
    N = len(key)
    for i in range(256):
        S[i] = i
        K[i] = key[(i % N)]
    
    j = 0
    for i in range(256):
        j = (j + S[i] + K[i]) % 256
        S[i], S[j] = S[j], S[i]
    return S

def RC4_Keystream_Generator(data_len, S):
    # RC4 Keystream Generator
    i = j = 0
    keystream = bytearray()
    for _ in range(data_len):
        i = (i + 1) % 256
        j = (j + S[i]) % 256
        S[i], S[j] = S[j], S[i]
        t = (S[i] + S[j]) % 256
        keystream.append(S[t]) 

    return keystream

def RC4(data, key):
    S = RC4_Init(key)
    keystream = RC4_Keystream_Generator(len(data), S)
    encrypted_data = bytearray()
    for d, k in zip(data, keystream):
        encrypted_data.append(d ^ k)
    return bytes(encrypted_data)


In [3]:
key = b"key"
plaintext = b"hello world"
print(f"Original text: {plaintext}")

# encrypt
ciphertext = RC4(plaintext, key)
print(f"Encrypted text: {ciphertext}")

# decrypt
decrypted = RC4(ciphertext, key)
print(f"Decrypted text: {decrypted}")

Original text: b'hello world'
Encrypted text: b"c\tX\x81K\xaf\x0c%:'l"
Decrypted text: b'hello world'


### The bias of the RC4 on the second byte

In [4]:
import secrets

def measure_second_byte(num_of_samples, key_length):
    counts = [0] * 256

    for _ in range(num_of_samples):
        key = secrets.token_bytes(key_length) # generating a random key 
        S = RC4_Init(key)
        ks = RC4_Keystream_Generator(2, S)
        second = ks[1] # second byte
        counts[second] += 1

    return counts

# run experiments
counts = measure_second_byte(100000, 16)

# results
total = sum(counts)
expected = 1 / 256

print(f"Total samples: {total}")
print(f"Expected uniform probability: {expected:.6f}")
print()

# probability for 0x00 (the biased one)
p0 = counts[0] / total
print(f"Probability of 0x00 = {p0:.6f}")

# compare to uniform
print(f"Bias factor: {p0 / expected:.2f}x more likely than uniform\n")

# show the top 5 most common bytes
top = sorted(range(256), key=lambda b: counts[b], reverse=True)[:5]
print("Top 5 most frequent bytes in the second byte:")
for b in top:
    print(f"  byte {b:02X} : p = {counts[b]/total:.6f}")

Total samples: 100000
Expected uniform probability: 0.003906

Probability of 0x00 = 0.007680
Bias factor: 1.97x more likely than uniform

Top 5 most frequent bytes in the second byte:
  byte 00 : p = 0.007680
  byte 75 : p = 0.004440
  byte EB : p = 0.004390
  byte D3 : p = 0.004380
  byte EF : p = 0.004360


### FMS attack for recovering the first byte of the WEP key

In [5]:
import random
from collections import Counter

def generate_weak_IV():
    # Weak IV = (3, 255, V), where V is related to the key byte index we're trying to recover
    V =  random.randrange(256)
    return [3, 255, V]

def generate_normal_IV():
    return [random.randrange(256) for _ in range(3)]

def packet_key(IV, long_term_key):
    return IV + long_term_key

def FMS_attack(num_of_packets, long_term_key_length, weak_ratio):
    long_term_key = [random.randrange(256) for _ in range(long_term_key_length)]
    real_K0 = long_term_key[0]

    guesses = []
    for _ in range(num_of_packets):
        # generating the IV
        use_weak = (random.random() < weak_ratio)
        if use_weak:
            IV = generate_weak_IV()
        else:
            IV = generate_normal_IV()

        # determining the true per packet key
        K = packet_key(IV, long_term_key)

        # generating the true first byte of the RC4 keystream
        S = RC4_Init(K)
        keystream0 = RC4_Keystream_Generator(1, S)[0]

        # encrypt known plaintext byte
        known_plaintext0 = random.randrange(256)
        known_ciphertext0 = known_plaintext0 ^ keystream0

        # attacker determines the first byte of keytream, knowing the first byte of the plaintext and ciphertext
        recovered_keystream0 = known_plaintext0 ^ known_ciphertext0

        if use_weak:
            guessed_K0 = (recovered_keystream0 - 6 - IV[2]) % 256
            guesses.append(guessed_K0)

    counts = Counter(guesses)
    best_guess, best_count = counts.most_common(1)[0]

    return {
        "long_term_key": long_term_key,
        "real_K0": real_K0,
        "best_guess": best_guess,
        "best_count": best_count,
        "counts": counts
    }

In [7]:
results = FMS_attack(10000, 5, 0.1)
print("True long_term_key:", results["long_term_key"])
print("True long_term_key[0]:", results["real_K0"])
print("Recovered long_term_key[0]:", results["best_guess"])
print("Top 5 guesses:", results["counts"].most_common(5))

True long_term_key: [119, 98, 88, 44, 233]
True long_term_key[0]: 119
Recovered long_term_key[0]: 119
Top 5 guesses: [(119, 28), (53, 20), (150, 19), (140, 19), (65, 18)]
